In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 20 teams with confirmed lineups


### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')

model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 4.5


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Scottie Barnes,Over,19.5,-137,2025-11-25,2025-11-24T22:55:11Z
1,Underdog,player_points,Scottie Barnes,Under,19.5,-137,2025-11-25,2025-11-24T22:55:11Z
2,Underdog,player_points,Donovan Mitchell,Over,30.5,-137,2025-11-25,2025-11-24T22:55:11Z
3,Underdog,player_points,Donovan Mitchell,Under,30.5,-137,2025-11-25,2025-11-24T22:55:11Z
4,Underdog,player_points,Brandon Ingram,Over,22.5,-137,2025-11-25,2025-11-24T22:55:11Z


## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs

Pre-computing predictions for 39 players...
Processing 35 players...
Generated 531 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
316,Jaden Ivey,Pelle Larsson,8.5,12.5,11.83,15.66,0.756,0.680,over,over,0,51.02,0.255,Low,High
104,Evan Mobley,Miles McBride,21.5,10.5,18.18,13.76,0.679,0.677,under,over,0,35.23,0.176,High,High
352,Bennedict Mathurin,Daniel Gafford,21.5,10.5,24.01,8.78,0.636,0.623,over,under,0,16.45,0.082,High,Med
374,Duncan Robinson,Davion Mitchell,10.5,10.5,12.75,12.29,0.635,0.613,over,over,0,14.34,0.072,High,High
319,Andrew Nembhard,Karl-Anthony Towns,16.5,23.5,18.46,21.57,0.607,0.605,over,under,0,7.93,0.040,High,High
436,Josh Hart,Simone Fontecchio,12.5,11.5,11.05,13.16,0.596,0.599,under,over,0,4.97,0.025,Med,High
163,Gradey Dick,Klay Thompson,8.5,10.5,9.78,12.01,0.591,0.587,over,over,0,2.10,0.011,Med,High
49,Donovan Mitchell,Tyrese Martin,30.5,8.5,29.14,9.83,0.573,0.586,under,over,0,-1.31,0.000,High,High
286,Cade Cunningham,Bam Adebayo,26.5,21.5,27.43,20.48,0.560,0.559,over,under,0,-7.96,0.000,High,High
490,Day'Ron Sharpe,D'Angelo Russell,6.5,12.5,7.11,13.42,0.548,0.550,over,over,0,-11.31,0.000,Low,High


### Prizepicks picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 50 players...
Processing 47 players...
Generated 949 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
238,Sandro Mamukelashvili,Jaden Ivey,8.5,8.5,-118,-145,11.86,11.83,over,over,0.721,0.756,0.5340,0.154,0.189,0.224,60.19,0.301,0,5.74,4.81,Med,Low,"(0.6, 23.1)","(2.4, 21.2)",0.05,0,60.2
748,Ben Sheppard,Pelle Larsson,6.5,12.5,-141,-110,9.53,15.66,over,over,0.693,0.680,0.4617,0.139,0.125,0.164,38.51,0.193,0,6.00,6.77,Med,High,"(0.0, 21.3)","(2.4, 28.9)",0.05,0,38.5
567,Tobias Harris,Miles McBride,11.5,10.5,-125,-130,14.55,13.76,over,over,0.662,0.677,0.4398,0.102,0.117,0.135,31.93,0.160,0,7.28,7.08,High,High,"(0.3, 28.8)","(0.0, 27.6)",0.05,0,31.9
810,Noah Clowney,Cooper Flagg,13.5,15.5,-118,-130,16.39,18.10,over,over,0.653,0.640,0.4097,0.100,0.087,0.112,22.90,0.115,0,7.35,7.25,High,High,"(2.0, 30.8)","(3.9, 32.3)",0.05,0,22.9
346,Collin Murray-Boyles,Ausar Thompson,6.5,11.0,-117,-137,8.39,13.49,over,over,0.640,0.639,0.4005,0.081,0.080,0.097,20.16,0.101,0,5.28,7.00,Med,High,"(0.0, 18.7)","(0.0, 27.2)",0.05,0,20.2
527,Bennedict Mathurin,Daniel Gafford,21.5,10.5,-127,-120,24.01,8.78,over,under,0.636,0.623,0.3882,0.084,0.070,0.091,16.45,0.082,0,7.22,5.51,High,Med,"(9.9, 38.2)","(0.0, 19.6)",0.05,0,16.5
154,Jaylon Tyson,Davion Mitchell,13.5,10.5,-127,-125,11.63,12.29,under,over,0.610,0.613,0.3661,0.052,0.055,0.063,9.83,0.049,0,6.71,6.25,High,High,"(0.0, 24.8)","(0.0, 24.5)",0.05,0,9.8
534,Andrew Nembhard,Karl-Anthony Towns,16.5,23.5,-130,-125,18.46,21.57,over,under,0.607,0.605,0.3598,0.046,0.045,0.053,7.93,0.040,0,7.26,7.23,High,High,"(4.2, 32.7)","(7.4, 35.7)",0.05,0,7.9
833,Jordan Clarkson,Simone Fontecchio,13.0,11.5,-137,-110,11.45,13.16,under,over,0.598,0.599,0.3511,0.047,0.048,0.055,5.33,0.027,0,6.24,6.64,High,High,"(0.0, 23.7)","(0.2, 26.2)",0.05,0,5.3
211,Gradey Dick,Josh Hart,8.5,12.5,-115,-122,9.78,11.05,over,under,0.591,0.596,0.3454,0.049,0.054,0.058,3.63,0.018,0,5.54,5.95,Med,Med,"(0.0, 20.6)","(0.0, 22.7)",0.05,0,3.6


## 3 leg parlay

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 39 players...
Processing 35 players...
Generated 6222 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
1721,Evan Mobley,Jaden Ivey,Pelle Larsson,21.5,8.5,12.5,18.18,11.83,15.66,0.679,0.756,0.680,under,over,over,0,88.35,0.177,High,Low,High
5012,Bennedict Mathurin,Miles McBride,Daniel Gafford,21.5,10.5,10.5,24.01,13.76,8.78,0.636,0.677,0.623,over,over,under,0,44.88,0.090,High,High,Med
5129,Duncan Robinson,Karl-Anthony Towns,Davion Mitchell,10.5,23.5,10.5,12.75,21.57,12.29,0.635,0.605,0.613,over,under,over,0,27.10,0.054,High,High,High
4764,Andrew Nembhard,Josh Hart,Simone Fontecchio,16.5,12.5,11.5,18.46,11.05,13.16,0.607,0.596,0.599,over,under,over,0,16.95,0.034,High,Med,High
2754,Gradey Dick,Tyrese Martin,Klay Thompson,8.5,8.5,10.5,9.78,9.83,12.01,0.591,0.586,0.587,over,over,over,0,9.87,0.020,Med,High,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=15)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 50 players...
Processing 47 players...
Generated 15390 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
5767,Sandro Mamukelashvili,Jaden Ivey,Pelle Larsson,8.5,8.5,12.5,11.86,11.83,15.66,0.721,0.756,0.680,over,over,over,0,100.00,0.200,Med,Low,High
13873,Ben Sheppard,Miles McBride,Cooper Flagg,6.5,10.5,15.5,9.53,13.76,18.10,0.693,0.677,0.640,over,over,over,0,62.27,0.125,Med,High,High
7770,Collin Murray-Boyles,Tobias Harris,Noah Clowney,6.5,11.5,13.5,8.39,14.55,16.39,0.640,0.662,0.653,over,over,over,0,49.43,0.099,Med,High,High
3301,Jaylon Tyson,Ausar Thompson,Daniel Gafford,13.5,11.0,10.5,11.63,13.49,8.78,0.610,0.639,0.623,under,over,under,0,31.00,0.062,High,High,Med
10461,Bennedict Mathurin,Andrew Nembhard,Davion Mitchell,21.5,16.5,10.5,24.01,18.46,12.29,0.636,0.607,0.613,over,over,over,0,27.69,0.055,High,High,High


In [18]:
# df = playerScoring('Trey Murphy III', s26, current_date, teamStarPlayer, projectedStartingFive)
# playerContext('Trey Murphy III', s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)